# ============================================================
# [Data Integration] Dental Offices in Berlin
# ============================================================

#### **Install dependencies (if not already installed):**
 - conda install -c conda-forge osmnx geopandas pandas
 - conda install -c conda-forge tqdm 
 - conda install -c conda-forge geopy

# ============================================================
# 0. Imports
# ============================================================

In [1056]:
# Core & Standard Library
from pathlib import Path
from time import sleep
import re

# Data Processing
import pandas as pd
import numpy as np

# Geospatial Processing
import geopandas as gpd
import osmnx as ox

from shapely.geometry import Point
from shapely.ops import unary_union

# Geocoding
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter
import pickle

# Utilities
from difflib import SequenceMatcher
from tqdm import tqdm

# Constants
WGS84 = "EPSG:4326"


# ============================================================
# 1. Data Extraction & Initial Inspection
# ============================================================


In [1057]:
# 1.1 OSM Settings
ox.settings.use_cache = True
ox.settings.log_console = True

# 1.2 Fetch Dental Offices from OpenStreetMap
tags = {"amenity": "dentist"}

dental_offices_osm = ox.features.features_from_place(
    "Berlin, Germany",
    tags=tags
)

print(f"Number of dental office entries fetched: {len(dental_offices_osm)}")
print(dental_offices_osm.head(3).T.to_string())
print(dental_offices_osm['healthcare:speciality'].value_counts())

Number of dental office entries fetched: 798
element                                                  node                                                                                                                                                   
id                                                  304183504                                                                       313539258                                                          325161442
geometry                         POINT (13.612096 52.5114112)                                                   POINT (13.3553052 52.5488382)                                      POINT (13.1804772 52.5088434)
addr:city                                              Berlin                                                                          Berlin                                                                NaN
addr:country                                               DE                                                          

/Users/alex/anaconda3/envs/ds/lib/python3.10/site-packages/shapely/predicates.py:878: RuntimeWarning: invalid value encountered in intersects
  return lib.intersects(a, b, **kwargs)
/Users/alex/anaconda3/envs/ds/lib/python3.10/site-packages/shapely/set_operations.py:451: RuntimeWarning: invalid value encountered in union
  return lib.union(a, b, **kwargs)


In [1058]:
# Save raw data for reproducibility
dental_offices_osm.to_csv("../sources/raw_osm_dental_offices_v_01_19_2026.csv", index=False)
dental_offices_osm.to_file("../sources/raw_osm_dental_offices_v_01_19_2026.geojson", driver="GeoJSON")

# Inspect dataset
dental_offices_osm.info()
print(dental_offices_osm.columns.tolist())

<class 'geopandas.geodataframe.GeoDataFrame'>
MultiIndex: 798 entries, ('node', 304183504) to ('way', 293129382)
Columns: 104 entries, geometry to type
dtypes: geometry(1), object(103)
memory usage: 689.8+ KB
['geometry', 'addr:city', 'addr:country', 'addr:housenumber', 'addr:postcode', 'addr:street', 'addr:suburb', 'amenity', 'health_facility:type', 'health_specialty:dentistry', 'healthcare', 'medical_system:western', 'office', 'operator', 'description', 'level', 'name', 'opening_hours', 'wheelchair', 'phone', 'toilets:wheelchair', 'website', 'contact:website', 'contact:email', 'contact:fax', 'contact:phone', 'check_date:opening_hours', 'check_date', 'email', 'fax', 'healthcare:speciality', 'air_conditioning', 'internet_access', 'internet_access:fee', 'payment:bank_transfer', 'payment:cash', 'payment:credit_cards', 'payment:debit_cards', 'payment:paypal', 'source', 'toilets', 'entrance', 'opening_hours:signed', 'wheelchair:description', 'emergency', 'name:de', 'name:en', 'payment:cont

# ============================================================
# 2. Data Selection & Column Standardization
# ============================================================

In [1059]:
# Select key columns relevant for dental officesс
columns = [
    "name",
    "addr:street",
    "addr:housenumber",
    "addr:postcode",
    "addr:city",
    "level",
    "opening_hours",
    "check_date",
    "healthcare:speciality",
    "wheelchair",
    "wheelchair:description",
    "phone",
    "email",
    "website",
    "geometry",
    "health_facility:type",
    "health_specialty:oral_surgery",
    "health_specialty:orthodontics",
    "health_specialty:periodontology"
]

# Filter the dataset to keep only the selected columns
dental_offices = dental_offices_osm[[c for c in columns if c in dental_offices_osm.columns]].copy()

# ============================================================
# Next Processing Steps (Roadmap for future PRs / scripts)
# ============================================================
# 1. Name normalization
#    - Standardize names such as "Zahnarztpraxis", "Drs.", and other practice naming conventions.
# 2. Address cleaning
#    - Format street names and house numbers to a consistent structure.
# 3. Category mapping
#    - Map specialization fields into a controlled vocabulary for consistency.
# 4. Deduplication logic
#    - Detect and handle overlaps between OSM entries and official city registries.


# ============================================================
# 3. Speciality Mapping
# ============================================================

In [1060]:
# Mapping of OSM health specialty columns to standardized speciality names
# Only consider values marked as "yes" or "main"
special_cols = {
    "health_specialty:oral_surgery": "oral_surgery",
    "health_specialty:orthodontics": "orthodontics",
    "health_specialty:periodontology": "periodontology"
}

def compute_speciality(row):
    """
    Determine standardized speciality for a dental office.
    
    Priority:
    1. Use 'healthcare:speciality' if non-empty
    2. Check boolean-style specialty columns ("yes" or "main")
    3. Default to "None"
    """
    val = row.get("healthcare:speciality")
    if pd.notna(val) and str(val).strip() != "":
        return str(val).strip()
    
    for col, name in special_cols.items():
        cell = row.get(col)
        if pd.notna(cell) and str(cell).lower() in ["yes", "main"]:
            return name
    
    return None

# Apply speciality mapping
dental_offices["speciality"] = dental_offices.apply(compute_speciality, axis=1)

# Drop original speciality columns to avoid redundancy
dental_offices.drop(
    columns=["healthcare:speciality"] + list(special_cols.keys()), inplace=True
)

dental_offices.head()

name     addr:street addr:housenumber  \
element id                                                                
node    304183504                  NaN  Hönower Straße               75   
        313539258  Zahnzentrum Wedding    Müllerstraße              34a   
        325161442             A. Nejad             NaN              NaN   
        345236220    Dr. Beate Lengert  Kurfürstendamm              218   
        391394177      Serpil Hartfiel  Kollwitzstraße               77   

                  addr:postcode addr:city level  \
element id                                        
node    304183504         12623    Berlin   NaN   
        313539258         13353    Berlin     2   
        325161442           NaN       NaN   NaN   
        345236220         10719    Berlin   NaN   
        391394177         10435    Berlin   NaN   

                                                       opening_hours  \
element id                                                             
node    304183504                                                NaN   
        313539258  Mo 09:00-19:00; Tu 09:00-18:00; We 09:00-17:00...   
        325161442  Mo-Tu 09:00-19:00; We 09:00-14:00; Th 09:00-19...   
        345236220                                                NaN   
        391394177  Mo,Tu,Th 08:00-19:00; We 18:00-18:00; Fr 08:00...   

                  check_date wheelchair wheelchair:description  \
element id                                                       
node    304183504        NaN        NaN                    NaN   
        313539258        NaN        yes                    NaN   
        325161442        NaN        yes                    NaN   
        345236220        NaN        NaN                    NaN   
        391394177        NaN         no                    NaN   

                              phone email                          website  \
element id                                                                   
node    304183504               NaN   NaN                              NaN   
        313539258               NaN   NaN                              NaN   
        325161442  +49 30 361 91 06   NaN                              NaN   
        345236220               NaN   NaN  http://www.dr-beate-lengert.de/   
        391394177               NaN   NaN                              NaN   

                                    geometry health_facility:type speciality  
element id                                                                    
node    304183504   POINT (13.6121 52.51141)               office       None  
        313539258  POINT (13.35531 52.54884)                  NaN       None  
        325161442  POINT (13.18048 52.50884)                  NaN       None  
        345236220  POINT (13.32814 52.50272)                  NaN       None  
        391394177  POINT (13.41899 52.53755)                  NaN       None

# ============================================================
# 4. Geometry Processing
# ============================================================

In [1061]:

# Ensure point geometry
dental_offices["geometry"] = dental_offices["geometry"].apply(
    lambda g: g if g.geom_type == "Point" else g.representative_point()
)

# Extract latitude and longitude
dental_offices["latitude"] = dental_offices.geometry.y
dental_offices["longitude"] = dental_offices.geometry.x

# Standardize address columns
dental_offices = dental_offices.rename(columns={
    "addr:street": "street",
    "addr:housenumber": "housenumber",
    "addr:postcode": "postcode",
    "addr:city": "city"
})
dental_offices.info()

<class 'geopandas.geodataframe.GeoDataFrame'>
MultiIndex: 798 entries, ('node', 304183504) to ('way', 293129382)
Data columns (total 18 columns):
 #   Column                  Non-Null Count  Dtype   
---  ------                  --------------  -----   
 0   name                    767 non-null    object  
 1   street                  582 non-null    object  
 2   housenumber             582 non-null    object  
 3   postcode                535 non-null    object  
 4   city                    527 non-null    object  
 5   level                   90 non-null     object  
 6   opening_hours           602 non-null    object  
 7   check_date              140 non-null    object  
 8   wheelchair              286 non-null    object  
 9   wheelchair:description  8 non-null      object  
 10  phone                   289 non-null    object  
 11  email                   88 non-null     object  
 12  website                 306 non-null    object  
 13  geometry                798 non-null   

# ============================================================
# 5. Neighborhood Assignment via Spatial Join
# ============================================================

In [1062]:

# Load neighborhoods GeoJSON (Berlin LOR Ortsteile)
lor_path = Path('../../mapping/lor_ortsteile.geojson')
neighborhoods = gpd.read_file(lor_path).to_crs("EPSG:4326")

# Rename for consistency
neighborhoods = neighborhoods.rename(columns={
    "BEZIRK": "district",
    "OTEIL": "neighborhood",
    "spatial_name": "neighborhood_id"
})

# Create GeoDataFrame for dental offices
dental_gdf = gpd.GeoDataFrame(dental_offices, geometry='geometry', crs='EPSG:4326')
print(f"✓ Created GeoDataFrame with {len(dental_gdf)} dental offices")


✓ Created GeoDataFrame with 798 dental offices


In [1063]:
# Spatial join to assign neighborhoods
df_with_districts = gpd.sjoin(
    dental_gdf,
    neighborhoods[["district", "neighborhood", "neighborhood_id", "geometry"]],
    how="left",
    predicate="within"
)

# Drop unnecessary columns from spatial join
df_final = df_with_districts.drop(columns=["index_right"])
df_final.head(3)

name          street housenumber postcode  \
element id                                                                    
node    304183504                  NaN  Hönower Straße          75    12623   
        313539258  Zahnzentrum Wedding    Müllerstraße         34a    13353   
        325161442             A. Nejad             NaN         NaN      NaN   

                     city level  \
element id                        
node    304183504  Berlin   NaN   
        313539258  Berlin     2   
        325161442     NaN   NaN   

                                                       opening_hours  \
element id                                                             
node    304183504                                                NaN   
        313539258  Mo 09:00-19:00; Tu 09:00-18:00; We 09:00-17:00...   
        325161442  Mo-Tu 09:00-19:00; We 09:00-14:00; Th 09:00-19...   

                  check_date wheelchair wheelchair:description  ... email  \
element id                                                      ...         
node    304183504        NaN        NaN                    NaN  ...   NaN   
        313539258        NaN        yes                    NaN  ...   NaN   
        325161442        NaN        yes                    NaN  ...   NaN   

                  website                   geometry health_facility:type  \
element id                                                                  
node    304183504     NaN   POINT (13.6121 52.51141)               office   
        313539258     NaN  POINT (13.35531 52.54884)                  NaN   
        325161442     NaN  POINT (13.18048 52.50884)                  NaN   

                  speciality   latitude  longitude             district  \
element id                                                                
node    304183504       None  52.511411  13.612096  Marzahn-Hellersdorf   
        313539258       None  52.548838  13.355305                Mitte   
        325161442       None  52.508843  13.180477              Spandau   

                   neighborhood neighborhood_id  
element id                                       
node    304183504     Mahlsdorf            1004  
        313539258       Wedding            0105  
        325161442  Wilhelmstadt            0509  

[3 rows x 21 columns]

# ============================================================
# 6. District ID Mapping
# ============================================================

In [1064]:
district_mapping = {
    'Mitte': '11001001',
    'Friedrichshain-Kreuzberg': '11002002',
    'Pankow': '11003003',
    'Charlottenburg-Wilmersdorf': '11004004',
    'Spandau': '11005005',
    'Steglitz-Zehlendorf': '11006006',
    'Tempelhof-Schöneberg': '11007007',
    'Neukölln': '11008008',
    'Treptow-Köpenick': '11009009',
    'Marzahn-Hellersdorf': '11010010',
    'Lichtenberg': '11011011',
    'Reinickendorf': '11012012'
}

df_final['district_id'] = df_final['district'].map(district_mapping).astype(str)

# Check for unmapped districts
unmapped = df_final[~df_final['district'].isin(district_mapping.keys())]['district'].unique()
if len(unmapped) > 0:
    print("⚠️ Unmapped districts found:", unmapped)

# ============================================================
# 7. Dental Office ID & Index Cleanup
# ============================================================

In [1065]:
df_final = df_final.reset_index()
df_final = df_final.drop(columns=["element"]).rename(columns={"id": "dental_office_id"})
df_final["dental_office_id"] = df_final["dental_office_id"].astype("string")
print(df_final.shape[0], "total dental office records after processing.")
print(df_final["dental_office_id"].nunique(), "unique dental offices IDs assigned.")
print(df_final.shape[0], "total dental office records after processing.")
print(df_final["dental_office_id"].nunique(), "unique dental offices IDs assigned.")
df_final.head(10)

798 total dental office records after processing.
798 unique dental offices IDs assigned.
798 total dental office records after processing.
798 unique dental offices IDs assigned.


,dental_office_id,name,street,housenumber,postcode,city,level,opening_hours,check_date,wheelchair,...,website,geometry,health_facility:type,speciality,latitude,longitude,district,neighborhood,neighborhood_id,district_id
0,304183504,NaN,Hönower Straße,75,12623,Berlin,NaN,NaN,NaN,NaN,...,NaN,POINT (13.6121 52.51141),office,None,52.511411,13.612096,Marzahn-Hellersdorf,Mahlsdorf,1004,11010010
1,313539258,Zahnzentrum Wedding,Müllerstraße,34a,13353,Berlin,2,Mo 09:00-19:00; Tu 09:00-18:00; We 09:00-17:00...,NaN,yes,...,NaN,POINT (13.35531 52.54884),NaN,None,52.548838,13.355305,Mitte,Wedding,0105,11001001
2,325161442,A. Nejad,NaN,NaN,NaN,NaN,NaN,Mo-Tu 09:00-19:00; We 09:00-14:00; Th 09:00-19...,NaN,yes,...,NaN,POINT (13.18048 52.50884),NaN,None,52.508843,13.180477,Spandau,Wilhelmstadt,0509,11005005
3,345236220,Dr. Beate Lengert,Kurfürstendamm,218,10719,Berlin,NaN,NaN,NaN,NaN,...,http://www.dr-beate-lengert.de/,POINT (13.32814 52.50272),NaN,None,52.502722,13.328137,Charlottenburg-Wilmersdorf,Charlottenburg,0401,11004004
4,391394177,Serpil Hartfiel,Kollwitzstraße,77,10435,Berlin,NaN,"Mo,Tu,Th 08:00-19:00; We 18:00-18:00; Fr 08:00...",NaN,no,...,NaN,POINT (13.41899 52.53755),NaN,None,52.537547,13.418994,Pankow,Prenzlauer Berg,0301,11003003
5,420517053,"Zahnärzte Nicolas Weiss, Volker Landmann",NaN,NaN,NaN,NaN,NaN,"Mo-Fr ""nach Vereinbarung""",NaN,NaN,...,NaN,POINT (13.40487 52.38497),NaN,None,52.384968,13.404870,Tempelhof-Schöneberg,Lichtenrade,0706,11007007
6,430545835,DentZ,Tempelhofer Damm,143,12099,Berlin,NaN,"Mo,Tu,Fr 09:00-16:00; We,Th 11:00-19:00",NaN,yes,...,NaN,POINT (13.38595 52.46631),NaN,None,52.466306,13.385948,Tempelhof-Schöneberg,Tempelhof,0703,11007007
7,442391661,Zahnklinik Medeco,NaN,NaN,NaN,NaN,NaN,"Mo-Fr 07:00-21:00; Sa,Su,PH 09:00-18:00",2023-08-04,limited,...,NaN,POINT (13.38518 52.45106),NaN,None,52.451063,13.385178,Tempelhof-Schöneberg,Mariendorf,0704,11007007
8,484267657,Mund-Kiefer-Gesichtschirugie,NaN,NaN,NaN,Berlin,NaN,NaN,NaN,NaN,...,NaN,POINT (13.31013 52.52516),NaN,None,52.525158,13.310129,Charlottenburg-Wilmersdorf,Charlottenburg,0401,11004004
9,552149348,Zahnärztliche Gemeinschaftspraxis,NaN,NaN,NaN,NaN,NaN,Mo-Th 08:00-19:00; Fr 08:00-14:00,NaN,NaN,...,https://www.zahnarztpraxis-speda.de/,POINT (13.35379 52.54138),NaN,None,52.541379,13.353790,Mitte,Wedding,0105,11001001


# ============================================================
# 8. Level / Floor Mapping
# ============================================================

In [1066]:
# Mapping DEU, UK und US
LEVEL_MAP_DEU = {
    "-1": "UG", "0": "EG", "1": "1.OG", "2": "2.OG", "3": "3.OG",
    "4": "4.OG", "5": "5.OG", "6": "6.OG", "7": "7.OG", "8": "8.OG", "9": "9.OG", "10": "10.OG"
}

LEVEL_MAP_UK = {
    "-1": "Basement", "0": "Ground Floor", "1": "First Floor", "2": "Second Floor",
    "3": "Third Floor", "4": "Fourth Floor", "5": "Fifth Floor", "6": "Sixth Floor"
}

LEVEL_MAP_US = {
    "-1": "Basement", "0": "First Floor", "1": "Second Floor", "2": "Third Floor",
    "3": "Fourth Floor", "4": "Fifth Floor", "5": "Sixth Floor", "6": "Seventh Floor"
}
# Select mapping (DEU / UK / US)
LEVEL_MAP = LEVEL_MAP_DEU  # Change to LEVEL_MAP_UK or LEVEL_MAP_US as needed

def format_level(level):
    """Convert numeric level to standardized floor label."""
    if pd.isna(level):
        return ""
    return LEVEL_MAP.get(str(level).strip(), level)




# ------------------------------------------------------------
# 9. Fallback Address Enrichment via Reverse Geocoding (Nominatim)
# ------------------------------------------------------------

In [1067]:
# -------------------------------------------------------------
# Initialize the Nominatim geocoder with a custom user agent
# Nominatim requires a user_agent string to identify the application.
# Using a descriptive name helps comply with their usage policy.
# -------------------------------------------------------------

geolocator = Nominatim(user_agent="berlin_dental_offices_project", timeout=10)
# RateLimiter: 1 request per second to respect public Nominatim policy
geocode_rate_limited = RateLimiter(geolocator.reverse, min_delay_seconds=1)

# -----------------------------
# Initialize local cache for reverse geocoding results
# -----------------------------
cache_file = "nominatim_reverse_cache.pkl"

try:
    with open(cache_file, "rb") as f:
        reverse_cache = pickle.load(f)
except FileNotFoundError:
    reverse_cache = {}

# -----------------------------
# Debug info before geocoding
# -----------------------------
print(f"[INFO] Missing house numbers before Nominatim: {df_final['housenumber'].isna().sum()}")
print(f"[INFO] Missing streets before Nominatim: {df_final['street'].isna().sum()}")

# -----------------------------
# Function: reverse geocode with shift, retry, and caching
# -----------------------------
def reverse_geocode_address_components(lat, lon, max_attempts=10, shift_deg=0.0001, retry_count=3):
    """
    Reverse-geocode latitude and longitude into structured address components.
    Uses radius search with small shifts and retries on timeout errors.
    Caches results to avoid repeated API calls.

    Parameters
    ----------
    lat, lon : float
        Coordinates to reverse-geocode
    max_attempts : int
        Number of shifted coordinate attempts (~10 meters each)
    shift_deg : float
        Degree shift applied for nearby search
    retry_count : int
        Number of retries per shifted coordinate in case of connection timeout

    Returns
    -------
    dict
        Dictionary with keys: street, housenumber, postcode, city
        Missing values returned as pd.NA
    """

    # Round coordinates for cache key (~0.1 m precision)
    key = (round(lat, 6), round(lon, 6))

    # Return cached result if available
    if key in reverse_cache:
        return reverse_cache[key]

    # Define coordinate shifts: original + small N/S/E/W shifts
    shifts = [(0, 0), (shift_deg, 0), (-shift_deg, 0), (0, shift_deg), (0, -shift_deg)]

    for dx, dy in shifts[:max_attempts]:
        for attempt in range(retry_count):
            try:
                # Perform reverse geocoding (rate-limited)
                location = geocode_rate_limited(
                    (lat + dx, lon + dy),
                    exactly_one=True,
                    language="en"
                )

                # If result found, extract structured address
                if location and location.raw.get("address"):
                    address = location.raw.get("address", {})

                    street_name = address.get("road") or address.get("place") or pd.NA
                    result = {
                        "street": street_name,
                        "housenumber": address.get("house_number", pd.NA),
                        "postcode": address.get("postcode", pd.NA),
                        "city": (
                            address.get("city")
                            or address.get("town")
                            or address.get("village")
                            or address.get("municipality")
                            or address.get("suburb")
                            or pd.NA
                        ),
                    }

                    # Save to cache and return
                    reverse_cache[key] = result
                    return result

            except Exception as e:
                # Warn and retry
                print(f"[WARN] Reverse geocode failed for ({lat+dx}, {lon+dy}), attempt {attempt+1}/{retry_count}: {e}")
                sleep(2)  # small pause before retry

    # If all attempts fail, store and return empty components
    result = {"street": pd.NA, "housenumber": pd.NA, "postcode": pd.NA, "city": pd.NA}
    reverse_cache[key] = result
    return result

# -----------------------------
# Apply reverse geocoding to missing address components
# -----------------------------
address_cols = ["street", "housenumber", "postcode", "city"]

# Mask: only rows with missing components and valid coordinates
mask = (
    df_final[address_cols].isna().any(axis=1) &
    df_final["latitude"].notna() &
    df_final["longitude"].notna()
)

# Iterate over masked rows
for idx, row in tqdm(df_final.loc[mask].iterrows(), total=mask.sum()):
    components = reverse_geocode_address_components(row["latitude"], row["longitude"])
    for col in address_cols:
        # Only fill missing values
        if pd.isna(row[col]) and pd.notna(components[col]):
            df_final.at[idx, col] = components[col]

# -----------------------------
# Save cache to disk for future runs
# -----------------------------
with open(cache_file, "wb") as f:
    pickle.dump(reverse_cache, f)

# -----------------------------
# Debug info after geocoding
# -----------------------------
print(f"[INFO] Missing house numbers after Nominatim: {df_final['housenumber'].isna().sum()}")
print(f"[INFO] Missing streets after Nominatim: {df_final['street'].isna().sum()}")
# -------------------------------------------------------------
# Note on the radius search:
# If no result is found at the original coordinates, the function
# shifts the coordinates slightly (~10 meters) in multiple directions
# to perform a nearby lookup.
#
# To maximize coverage around each point, the parameter `max_attempts`
# was set to 10, allowing the function to probe multiple nearby locations
# in order to increase the likelihood of finding an associated address.
#
# This approach helps capture points that lie between buildings or street
# geometries and would otherwise not be resolved by Nominatim.
#
# Additionally, if no 'road' field is returned by Nominatim, the function
# falls back to using the 'place' field to populate the 'street' value,
# ensuring that addresses stored under alternative address tags are
# also captured.
# -------------------------------------------------------------


[INFO] Missing house numbers before Nominatim: 216
[INFO] Missing streets before Nominatim: 216


100%|██████████| 280/280 [04:52<00:00,  1.04s/it]

[INFO] Missing house numbers after Nominatim: 187
[INFO] Missing streets after Nominatim: 1


# ============================================================
# 10. Address Formatting
# ============================================================

In [1068]:
def format_address(row):
    """
    Construct full address string with optional floor and neighborhood.
    If all core address fields are missing, return 'Unknown'.
    """
    # If all core fields are missing
    if all(pd.isna(row[col]) for col in ['street', 'housenumber', 'level', 'postcode', 'city']):
        return pd.NA
    
    # Prepare components
    street = row['street'] if pd.notna(row['street']) else ""
    housenumber = row['housenumber'] if pd.notna(row['housenumber']) else ""
    postcode = row['postcode'] if pd.notna(row['postcode']) else ""
    city = row['city'] if pd.notna(row['city']) else ""
    neighborhood = row['neighborhood'] if pd.notna(row['neighborhood']) else ""
    
    # Floor formatting
    level_str = format_level(row['level'])
    
    # Construct address parts
    parts = [f"{street} {housenumber}".strip()]
    if level_str:
        parts.append(level_str)
    city_line = f"{postcode} {city}".strip()
    if neighborhood:
        city_line += f"-{neighborhood}"
    parts.append(city_line)
    
    return ", ".join([p for p in parts if p])

# Apply formatted address
df_final["address"] = df_final.apply(format_address, axis=1)
# Quick check of results
df_final["address"].isna().sum()
df_final[["latitude", "longitude", "address"]].head(10)



,latitude,longitude,address
0,52.511411,13.612096,"Hönower Straße 75, 12623 Berlin-Mahlsdorf"
1,52.548838,13.355305,"Müllerstraße 34a, 2.OG, 13353 Berlin-Wedding"
2,52.508843,13.180477,"Gatower Straße, 13595 Berlin-Wilhelmstadt"
3,52.502722,13.328137,"Kurfürstendamm 218, 10719 Berlin-Charlottenburg"
4,52.537547,13.418994,"Kollwitzstraße 77, 10435 Berlin-Prenzlauer Berg"
5,52.384968,13.404870,"Goltzstraße, 12307 Berlin-Lichtenrade"
6,52.466306,13.385948,"Tempelhofer Damm 143, 12099 Berlin-Tempelhof"
7,52.451063,13.385178,"Mariendorfer Damm, 12109 Berlin-Mariendorf"
8,52.525158,13.310129,"Ilsenburger Straße, 10589 Berlin-Charlottenburg"
9,52.541379,13.353790,"Sprengelstraße, 13353 Berlin-Wedding"


In [1069]:
df_final[df_final['street'].isna()] 

,dental_office_id,name,street,housenumber,postcode,city,level,opening_hours,check_date,wheelchair,...,geometry,health_facility:type,speciality,latitude,longitude,district,neighborhood,neighborhood_id,district_id,address
131,3353229503,Dentalzentrum Pankow,NaN,1,13187,Berlin,NaN,"Mo-We 10:30-18:00; Th,Fr 08:30-16:00",NaN,yes,...,POINT (13.41094 52.56735),NaN,None,52.567345,13.41094,Pankow,Pankow,0307,11003003,"1, 13187 Berlin-Pankow"


In [1070]:
# Hardcoding the street names for entries with missing street information
hardcoded_streets = "Garbátyplatz"
df_final.loc[df_final['name'] == "Dentalzentrum Pankow", 'street'] = hardcoded_streets

In [1071]:
df_final[df_final['street'].isna()] 

,dental_office_id,name,street,housenumber,postcode,city,level,opening_hours,check_date,wheelchair,...,geometry,health_facility:type,speciality,latitude,longitude,district,neighborhood,neighborhood_id,district_id,address


# ============================================================
# 11. Data Overview / Quality Check (Columns, Types, Missing Values)
# ============================================================

In [1072]:
# Missing Values
missing = df_final.isna().sum().sort_values(ascending=False)
missing_pct = (missing / len(df_final) * 100).round(1)
print('Data types:')
display(df_final.dtypes)
print(f'Dental Offices Data: Rows: {df_final.shape[0]}, Columns: {df_final.shape[1]}')
print("Missing Values Summary:")
pd.DataFrame({
    "missing_count": missing,
    "missing_pct": missing_pct
})

Data types:


dental_office_id          string[python]
name                              object
street                            object
housenumber                       object
postcode                          object
city                              object
level                             object
opening_hours                     object
check_date                        object
wheelchair                        object
wheelchair:description            object
phone                             object
email                             object
website                           object
geometry                        geometry
health_facility:type              object
speciality                        object
latitude                         float64
longitude                        float64
district                          object
neighborhood                      object
neighborhood_id                   object
district_id                       object
address                           object
dtype: object

Dental Offices Data: Rows: 798, Columns: 24
Missing Values Summary:


,missing_count,missing_pct
wheelchair:description,790,99.0
health_facility:type,775,97.1
email,710,89.0
level,708,88.7
speciality,686,86.0
check_date,658,82.5
wheelchair,512,64.2
phone,509,63.8
website,492,61.7
opening_hours,196,24.6
